In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load processed dataset
data = pd.read_csv("../data/processed/processed_co2_data.csv")
print("Data Shape:", data.shape)
data.head()


Data Shape: (18646, 8)


,entity,code,year,annual co₂ emissions (tonnes ),Emission_Intensity,Renewable_Share,missing_emission_intensity,missing_renewable_share
0,Afghanistan,AFG,1949,14656.0,3.622849,33.309697,False,False
1,Afghanistan,AFG,1950,84272.0,3.689608,33.623736,False,False
2,Afghanistan,AFG,1951,91600.0,3.695561,36.484803,False,False
3,Afghanistan,AFG,1952,91600.0,3.676920,36.304646,False,False
4,Afghanistan,AFG,1953,106256.0,3.591820,33.986926,False,False


In [2]:
# Verify data is loaded and not empty
print("Data shape:", data.shape)
print("Available columns:", data.columns.tolist())
print("Data dtypes:\n", data.dtypes)

if data.shape[0] == 0:
	print("WARNING: Data is empty. Check file path or data content.")
	import os
	file_path = "../data/processed/processed_co2_data.csv"
	print(f"File exists: {os.path.exists(file_path)}")

# Replace with your actual column names
target_col = "annual co₂ emissions (tonnes )"
if target_col not in data.columns:
	raise KeyError(f"Target column '{target_col}' not found in data. Available columns: {list(data.columns)}")

# Convert target column to numeric
data[target_col] = pd.to_numeric(data[target_col], errors='coerce')

# Drop rows with missing target values
data = data.dropna(subset=[target_col])
print(f"Data shape after cleaning: {data.shape}")

X = data.drop(columns=[target_col])
y = data[target_col]

if X.shape[0] > 0:
	# Split data
	X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
	print("Train Shape:", X_train.shape, " | Test Shape:", X_test.shape)
else:
	print("ERROR: No valid data available for training after cleaning.")


Data shape: (18646, 8)
Available columns: ['entity', 'code', 'year', 'annual co₂ emissions (tonnes )', 'Emission_Intensity', 'Renewable_Share', 'missing_emission_intensity', 'missing_renewable_share']
Data dtypes:
 entity                             object
code                               object
year                                int64
annual co₂ emissions (tonnes )    float64
Emission_Intensity                float64
Renewable_Share                   float64
missing_emission_intensity           bool
missing_renewable_share              bool
dtype: object
Data shape after cleaning: (18646, 8)
Train Shape: (14916, 7)  | Test Shape: (3730, 7)


In [3]:
# Ensure training data exists and is non-empty before fitting
if 'X_train' in globals() and 'y_train' in globals() and hasattr(X_train, "shape") and X_train.shape[0] > 0:
	# Handle missing values by filling them with 0, as LinearRegression cannot handle NaNs.
	X_train = X_train.fillna(0)
	X_test = X_test.fillna(0)

	# Encode categorical columns to numeric because ML models require numeric input.
	for col in X_train.select_dtypes(include=['object']).columns:
		# We combine train and test to create a consistent mapping for all categories.
		all_values = pd.concat([X_train[col], X_test[col]]).astype(str).unique()
		mapping = {val: i for i, val in enumerate(all_values)}
		X_train[col] = X_train[col].astype(str).map(mapping)
		X_test[col] = X_test[col].astype(str).map(mapping)

	lr = LinearRegression()
	lr.fit(X_train, y_train)
	
	# Only predict if X_test exists and is non-empty
	if 'X_test' in globals() and hasattr(X_test, "shape") and X_test.shape[0] > 0:
		y_pred_lr = lr.predict(X_test)
		print("Linear Regression trained. Predicted on X_test with shape:", X_test.shape)
		
		# Evaluate the model
		print("\nLinear Regression Metrics:")
		print(f"  MAE: {mean_absolute_error(y_test, y_pred_lr)}")
		print(f"  MSE: {mean_squared_error(y_test, y_pred_lr)}")
		print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lr))}")
		print(f"  R2 Score: {r2_score(y_test, y_pred_lr)}")
	else:
		y_pred_lr = None
		print("Linear Regression trained but X_test unavailable or empty. Skipping prediction.")
else:
	y_pred_lr = None
	print("Skipping Linear Regression training: X_train/y_train not available or empty.")


Linear Regression trained. Predicted on X_test with shape: (3730, 7)

Linear Regression Metrics:
  MAE: 278535599.05188036
  MSE: 1.5739324135483346e+18
  RMSE: 1254564631.0765877
  R2 Score: 0.016190134563130676


In [4]:
# Ensure training data exists and is non-empty before fitting
if 'X_train' in globals() and 'y_train' in globals() and hasattr(X_train, "shape") and X_train.shape[0] > 0:
	rf = RandomForestRegressor(n_estimators=200, random_state=42)
	rf.fit(X_train, y_train)
	
	# Only predict if X_test exists and is non-empty
	if 'X_test' in globals() and hasattr(X_test, "shape") and X_test.shape[0] > 0:
		y_pred_rf = rf.predict(X_test)
		print("Random Forest trained. Predicted on X_test with shape:", X_test.shape)
	else:
		y_pred_rf = None
		print("Random Forest trained but X_test unavailable or empty. Skipping prediction.")
else:
	y_pred_rf = None
	print("Skipping Random Forest training: X_train/y_train not available or empty.")


Random Forest trained. Predicted on X_test with shape: (3730, 7)


In [5]:
# Ensure necessary modules/objects are available in this cell (import only if missing)
if 'pd' not in globals():
    import pandas as pd
if 'np' not in globals():
    import numpy as np
if 'train_test_split' not in globals():
    from sklearn.model_selection import train_test_split
if 'RandomForestRegressor' not in globals():
    from sklearn.ensemble import RandomForestRegressor
if 'mean_absolute_error' not in globals():
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import os

# 0) Ensure dataset is loaded (either use existing `data` or load from file)
data_empty = False
if 'data' not in globals() or getattr(data, "shape", (0,))[0] == 0:
    file_path = "../data/processed/processed_co2_data.csv"
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File does not exist: {file_path}")
    data = pd.read_csv(file_path)
    print("Loaded data from file:", file_path, "shape:", data.shape)
    if data.shape[0] == 0:
        # Do not raise here; handle empty dataset gracefully so notebook can continue
        print("WARNING: Loaded dataset has 0 rows. Skipping preprocessing and model training.")
        print("Columns in file:", data.columns.tolist())
        data_empty = True
else:
    print("Using existing 'data' with shape:", getattr(data, "shape", None))

if data_empty:
    # Provide safe placeholders so later cells that reference X, y won't error out.
    target = "annual co₂ emissions (tonnes )"
    X = pd.DataFrame(columns=[c for c in getattr(data, "columns", []) if c != target])
    y = pd.Series(dtype=float)
    # Also set train/test placeholders
    X_train = pd.DataFrame(columns=X.columns)
    X_test = pd.DataFrame(columns=X.columns)
    y_train = pd.Series(dtype=float)
    y_test = pd.Series(dtype=float)
    print("Exiting preprocessing/training steps due to empty dataset.")
else:
    # 1) Set target and features
    target = "annual co₂ emissions (tonnes )"
    if target not in data.columns:
        raise KeyError(f"Target column '{target}' not found in data.columns: {data.columns.tolist()}")

    X = data.drop(columns=[target]).copy()
    y = data[target].copy()

    # 2) Quick NA check (you can change behavior if you prefer imputation)
    print("Missing values per column:\n", X.isna().sum())
    # If you want to drop rows with missing target:
    mask = y.notna()
    X = X[mask]
    y = y[mask]
    print("After dropping rows with missing target, shapes:", X.shape, y.shape)

    # 3) Encode categorical columns (only object/category dtype)
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    if cat_cols:
        print("Encoding categorical columns:", cat_cols)
        for col in cat_cols:
            # Convert to string to avoid issues with mixed types / NaNs
            X[col] = X[col].astype(str)
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col])

    # 4) Ensure X, y are numeric and have rows
    n_samples = X.shape[0]
    if n_samples == 0:
        # No samples left after preprocessing: set placeholders and skip training
        print("WARNING: No samples left after preprocessing. Skipping model training.")
        X_train = pd.DataFrame(columns=X.columns)
        X_test = pd.DataFrame(columns=X.columns)
        y_train = pd.Series(dtype=float)
        y_test = pd.Series(dtype=float)
    else:
        # 5) Choose a safe test_size that guarantees at least 1 sample in train and test
        test_count = max(1, int(np.ceil(0.2 * n_samples)))
        train_count = n_samples - test_count
        if train_count < 1:
            # If dataset very small, use leave-one-out-like: keep 1 test sample
            test_count = 1
            train_count = n_samples - 1
        test_size = test_count / n_samples

        print(f"n_samples={n_samples}, train_count={train_count}, test_count={test_count}, test_size={test_size:.3f}")

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)

        print("X_train.shape:", X_train.shape, "X_test.shape:", X_test.shape)

# 6) Train RandomForest only if we have training samples
if 'X_train' in globals() and hasattr(X_train, "shape") and X_train.shape[0] > 0 and \
   'y_train' in globals() and hasattr(y_train, "shape") and y_train.shape[0] > 0:
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    print("Random Forest trained.")
    if 'X_test' in globals() and hasattr(X_test, "shape") and X_test.shape[0] > 0:
        y_pred = rf.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        print(f"Evaluation on test set -> MAE: {mae:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}")
        # Feature importances (sorted)
        fi = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
        print("Top features:\n", fi.head(10))
    else:
        print("No X_test rows — trained but cannot evaluate.")
else:
    print("Skipping Random Forest training: no training samples available (dataset empty or all rows filtered).")


Using existing 'data' with shape: (18646, 8)
Missing values per column:
 entity                        0
code                          0
year                          0
Emission_Intensity            0
Renewable_Share               0
missing_emission_intensity    0
missing_renewable_share       0
dtype: int64
After dropping rows with missing target, shapes: (18646, 7) (18646,)
Encoding categorical columns: ['entity', 'code']
n_samples=18646, train_count=14916, test_count=3730, test_size=0.200
X_train.shape: (14916, 7) X_test.shape: (3730, 7)
Random Forest trained.
Evaluation on test set -> MAE: 10571441.6276, RMSE: 69767781.0745, R2: 0.9970
Top features:
 year                          0.539564
entity                        0.246601
code                          0.122172
Emission_Intensity            0.049078
Renewable_Share               0.042584
missing_emission_intensity    0.000000
missing_renewable_share       0.000000
dtype: float64


In [6]:
import pickle
from pathlib import Path

# CHANGE THIS PATH TO YOUR FLASK APP FOLDER
BASE_DIR = Path(r"C:\Users\Giri V\Downloads\CO2_Emission_Estimation_Full\CO2_Emission_Estimation_Full\flask_app")

MODEL_PATH = BASE_DIR / "models" / "co2_model.pkl"
SCALER_PATH = BASE_DIR / "models" / "scaler.pkl"
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

# save trained model
with open(MODEL_PATH, "wb") as f:
    pickle.dump(rf, f)

# The variable 'scaler' is not defined. The following lines are commented out.
# with open(SCALER_PATH, "wb") as f:
#     pickle.dump(scaler, f)

print("Saved model to:", MODEL_PATH)
# print("Saved scaler to:", SCALER_PATH)


Saved model to: C:\Users\Giri V\Downloads\CO2_Emission_Estimation_Full\CO2_Emission_Estimation_Full\flask_app\models\co2_model.pkl


In [7]:
# create_valid_scaler.py
"""
Create a valid StandardScaler pickle for the Flask app.
This scaler is fitted on a small dummy dataset with the same
feature order used by prepare_features_from_form():
["year", "Emission_Intensity", "Renewable_Share", "population"]
"""
from pathlib import Path
import pickle
import numpy as np
from sklearn.preprocessing import StandardScaler

# ADJUST THIS PATH if your flask_app is elsewhere
BASE = Path(r"C:\Users\Giri V\Downloads\CO2_Emission_Estimation_Full\CO2_Emission_Estimation_Full\flask_app")
MODELS_DIR = BASE / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

SCALER_PATH = MODELS_DIR / "scaler.pkl"

# Create dummy training data (4 features):
# year, Emission_Intensity, Renewable_Share, population
X_dummy = np.array([
    [2000, 5.0, 10.0, 1_000_000],
    [2005, 4.8, 12.0, 1_050_000],
    [2010, 4.6, 14.0, 1_100_000],
    [2015, 4.4, 16.0, 1_150_000],
    [2020, 4.2, 18.0, 1_200_000],
], dtype=float)

scaler = StandardScaler()
scaler.fit(X_dummy)

# Save the scaler
with open(SCALER_PATH, "wb") as f:
    pickle.dump(scaler, f)

print("Saved scaler to:", SCALER_PATH)
print("Scaler mean:", scaler.mean_)
print("Scaler scale:", scaler.scale_)


Saved scaler to: C:\Users\Giri V\Downloads\CO2_Emission_Estimation_Full\CO2_Emission_Estimation_Full\flask_app\models\scaler.pkl
Scaler mean: [2.01e+03 4.60e+00 1.40e+01 1.10e+06]
Scaler scale: [7.07106781e+00 2.82842712e-01 2.82842712e+00 7.07106781e+04]
